Cell 1 — install packages if needed

In [1]:
import sys
!{sys.executable} -m pip install joblib scikit-learn pandas numpy


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\PR\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Cell 2 — imports

In [2]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

Cell 3 — project paths

In [4]:
from pathlib import Path

BASE_DIR = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_PATH = BASE_DIR / "data" / "nlp" / "workout_logs_labeled.csv"
MODEL_DIR = BASE_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR =", BASE_DIR)
print("DATA_PATH =", DATA_PATH)
print("DATA_PATH exists? =", DATA_PATH.exists())
print("MODEL_DIR =", MODEL_DIR)

BASE_DIR = D:\OneDrive - FORSAN FOODS & CONSUMER PRODUCTS\AI Sandiego\Capstone_Project\fitness-journey-analyzer
DATA_PATH = D:\OneDrive - FORSAN FOODS & CONSUMER PRODUCTS\AI Sandiego\Capstone_Project\fitness-journey-analyzer\data\nlp\workout_logs_labeled.csv
DATA_PATH exists? = True
MODEL_DIR = D:\OneDrive - FORSAN FOODS & CONSUMER PRODUCTS\AI Sandiego\Capstone_Project\fitness-journey-analyzer\models


In [8]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_PATH = BASE_DIR / "data" / "nlp" / "workout_logs_labeled.csv"

df = pd.read_csv(DATA_PATH)

print(df.shape)
df.head()

(40, 3)


,workout_text,workout_type,fatigue_level
0,45 min running felt great,cardio,low
1,30 min cycling workout,cardio,low
2,20 min jogging and stretching,cardio,low
3,ran 5km and felt strong,cardio,low
4,cardio session treadmill 40 min,cardio,medium


Cell 4 — load labeled workout data

In [9]:
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Shape: (40, 3)
Columns: ['workout_text', 'workout_type', 'fatigue_level']


,workout_text,workout_type,fatigue_level
0,45 min running felt great,cardio,low
1,30 min cycling workout,cardio,low
2,20 min jogging and stretching,cardio,low
3,ran 5km and felt strong,cardio,low
4,cardio session treadmill 40 min,cardio,medium


Cell 5 — clean data

In [10]:
df = df.dropna(subset=["workout_text", "workout_type", "fatigue_level"]).copy()

df["workout_text"] = df["workout_text"].astype(str).str.strip()
df["workout_type"] = df["workout_type"].astype(str).str.strip().str.lower()
df["fatigue_level"] = df["fatigue_level"].astype(str).str.strip().str.lower()

df = df[df["workout_text"] != ""]

print("Cleaned shape:", df.shape)
df.head()

Cleaned shape: (40, 3)


,workout_text,workout_type,fatigue_level
0,45 min running felt great,cardio,low
1,30 min cycling workout,cardio,low
2,20 min jogging and stretching,cardio,low
3,ran 5km and felt strong,cardio,low
4,cardio session treadmill 40 min,cardio,medium


Cell 6 — label distribution

In [11]:
print("Workout type distribution:")
print(df["workout_type"].value_counts())
print()

print("Fatigue level distribution:")
print(df["fatigue_level"].value_counts())

Workout type distribution:
workout_type
cardio      10
strength    10
mobility    10
recovery    10
Name: count, dtype: int64

Fatigue level distribution:
fatigue_level
low       23
medium    10
high       7
Name: count, dtype: int64


Part A — Workout Type Model
Cell 7 — split for workout type

In [12]:
X = df["workout_text"]
y_type = df["workout_type"]

X_train_type, X_test_type, y_train_type, y_test_type = train_test_split(
    X,
    y_type,
    test_size=0.2,
    random_state=42,
    stratify=y_type
)

print("Workout type train size:", len(X_train_type))
print("Workout type test size:", len(X_test_type))

Workout type train size: 32
Workout type test size: 8


Cell 8 — train workout type model

In [13]:
workout_type_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        max_features=5000
    )),
    ("clf", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

workout_type_pipeline.fit(X_train_type, y_train_type)
print("Workout type model training completed.")

Workout type model training completed.


Cell 9 — evaluate workout type model

In [14]:
y_pred_type = workout_type_pipeline.predict(X_test_type)

print("Workout Type Accuracy:", accuracy_score(y_test_type, y_pred_type))
print()
print(classification_report(y_test_type, y_pred_type))

Workout Type Accuracy: 0.75

              precision    recall  f1-score   support

      cardio       0.00      0.00      0.00         2
    mobility       0.67      1.00      0.80         2
    recovery       1.00      1.00      1.00         2
    strength       0.67      1.00      0.80         2

    accuracy                           0.75         8
   macro avg       0.58      0.75      0.65         8
weighted avg       0.58      0.75      0.65         8



C:\Users\PR\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\PR\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\PR\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classificatio

Cell 10 — test workout type manually

In [15]:
sample_texts = [
    "45 min running and cardio session felt great",
    "heavy bench press and shoulder press today",
    "light yoga stretching and mobility work",
    "rest day and body still tired",
    "deadlift and squat session"
]

preds = workout_type_pipeline.predict(sample_texts)
probs = workout_type_pipeline.predict_proba(sample_texts)

for text, pred, prob in zip(sample_texts, preds, probs):
    print("TEXT:", text)
    print("PREDICTED WORKOUT TYPE:", pred)
    print("CONFIDENCE:", round(max(prob), 4))
    print("-" * 60)

TEXT: 45 min running and cardio session felt great
PREDICTED WORKOUT TYPE: cardio
CONFIDENCE: 0.481
------------------------------------------------------------
TEXT: heavy bench press and shoulder press today
PREDICTED WORKOUT TYPE: strength
CONFIDENCE: 0.3861
------------------------------------------------------------
TEXT: light yoga stretching and mobility work
PREDICTED WORKOUT TYPE: mobility
CONFIDENCE: 0.5081
------------------------------------------------------------
TEXT: rest day and body still tired
PREDICTED WORKOUT TYPE: recovery
CONFIDENCE: 0.5883
------------------------------------------------------------
TEXT: deadlift and squat session
PREDICTED WORKOUT TYPE: strength
CONFIDENCE: 0.4409
------------------------------------------------------------


Part B — Fatigue Level Model
Cell 11 — split for fatigue model

In [16]:
y_fatigue = df["fatigue_level"]

X_train_fatigue, X_test_fatigue, y_train_fatigue, y_test_fatigue = train_test_split(
    X,
    y_fatigue,
    test_size=0.2,
    random_state=42,
    stratify=y_fatigue
)

print("Fatigue train size:", len(X_train_fatigue))
print("Fatigue test size:", len(X_test_fatigue))

Fatigue train size: 32
Fatigue test size: 8


Cell 12 — train fatigue model

In [17]:
fatigue_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        max_features=5000
    )),
    ("clf", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

fatigue_pipeline.fit(X_train_fatigue, y_train_fatigue)
print("Fatigue model training completed.")

Fatigue model training completed.


Cell 13 — evaluate fatigue model

In [18]:
y_pred_fatigue = fatigue_pipeline.predict(X_test_fatigue)

print("Fatigue Accuracy:", accuracy_score(y_test_fatigue, y_pred_fatigue))
print()
print(classification_report(y_test_fatigue, y_pred_fatigue))

Fatigue Accuracy: 0.625

              precision    recall  f1-score   support

        high       0.00      0.00      0.00         1
         low       0.62      1.00      0.77         5
      medium       0.00      0.00      0.00         2

    accuracy                           0.62         8
   macro avg       0.21      0.33      0.26         8
weighted avg       0.39      0.62      0.48         8



C:\Users\PR\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\PR\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\PR\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classificatio

Cell 14 — test fatigue model manually

In [19]:
sample_texts = [
    "felt strong and energetic after workout",
    "legs are sore and very tired",
    "normal gym session today",
    "completely exhausted after deadlifts",
    "good session and body feels fresh"
]

preds = fatigue_pipeline.predict(sample_texts)
probs = fatigue_pipeline.predict_proba(sample_texts)

for text, pred, prob in zip(sample_texts, preds, probs):
    print("TEXT:", text)
    print("PREDICTED FATIGUE LEVEL:", pred)
    print("CONFIDENCE:", round(max(prob), 4))
    print("-" * 60)

TEXT: felt strong and energetic after workout
PREDICTED FATIGUE LEVEL: low
CONFIDENCE: 0.6329
------------------------------------------------------------
TEXT: legs are sore and very tired
PREDICTED FATIGUE LEVEL: low
CONFIDENCE: 0.4638
------------------------------------------------------------
TEXT: normal gym session today
PREDICTED FATIGUE LEVEL: low
CONFIDENCE: 0.6004
------------------------------------------------------------
TEXT: completely exhausted after deadlifts
PREDICTED FATIGUE LEVEL: low
CONFIDENCE: 0.5169
------------------------------------------------------------
TEXT: good session and body feels fresh
PREDICTED FATIGUE LEVEL: low
CONFIDENCE: 0.591
------------------------------------------------------------


Save both models
Cell 15 — save models

In [20]:
WORKOUT_MODEL_PATH = MODEL_DIR / "workout_type_model.pkl"
FATIGUE_MODEL_PATH = MODEL_DIR / "fatigue_model.pkl"

joblib.dump(workout_type_pipeline, WORKOUT_MODEL_PATH)
joblib.dump(fatigue_pipeline, FATIGUE_MODEL_PATH)

print("Saved workout type model to:", WORKOUT_MODEL_PATH)
print("Exists?", WORKOUT_MODEL_PATH.exists())

print("Saved fatigue model to:", FATIGUE_MODEL_PATH)
print("Exists?", FATIGUE_MODEL_PATH.exists())

Saved workout type model to: D:\OneDrive - FORSAN FOODS & CONSUMER PRODUCTS\AI Sandiego\Capstone_Project\fitness-journey-analyzer\models\workout_type_model.pkl
Exists? True
Saved fatigue model to: D:\OneDrive - FORSAN FOODS & CONSUMER PRODUCTS\AI Sandiego\Capstone_Project\fitness-journey-analyzer\models\fatigue_model.pkl
Exists? True


Reload test
Cell 16 — reload and test saved models

In [21]:
loaded_workout_model = joblib.load(MODEL_DIR / "workout_type_model.pkl")
loaded_fatigue_model = joblib.load(MODEL_DIR / "fatigue_model.pkl")

test_text = ["heavy squat workout and legs are sore"]

workout_pred = loaded_workout_model.predict([test_text[0]])[0]
fatigue_pred = loaded_fatigue_model.predict([test_text[0]])[0]

workout_conf = max(loaded_workout_model.predict_proba([test_text[0]])[0])
fatigue_conf = max(loaded_fatigue_model.predict_proba([test_text[0]])[0])

print("Workout Type:", workout_pred, "Confidence:", round(workout_conf, 4))
print("Fatigue Level:", fatigue_pred, "Confidence:", round(fatigue_conf, 4))

Workout Type: strength Confidence: 0.3926
Fatigue Level: high Confidence: 0.4336
